# E0 · Línea base con validación cruzada repetida
**Parte de:** `TweetToxic_modelo.ipynb` (stacking de 5 modelos, Kaggle público **0,79160**).

**Objetivo.** Reproducir exactamente el modelo que ya se envió y medirlo con **3 semillas** de validación cruzada en vez de una sola. Esta es la referencia contra la que se compara cada prueba (E1, E2, ...).

**Por qué.** Las diferencias entre configuraciones son de 0,3-0,6 puntos, y el leaderboard público (~577 filas) tiene un ruido de ±2,5 puntos. Con una sola partición de la CV, una mejora puede ser suerte. Promediar 3 semillas y mirar la desviación permite distinguir mejoras reales de ruido.

**Criterio de adopción (para todas las pruebas):** se adopta un cambio si sube el F1 medio en ≥ 0,003 y mejora en al menos 2 de las 3 semillas.

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **CNB** | *Complement Naive Bayes*: clasificador probabilístico simple por conteo de palabras |
| **LR** | Regresión logística: clasificador lineal con probabilidades |
| **SVM** | Máquina de vectores de soporte lineal, calibrada para dar probabilidades |
| **NB-LR** | Regresión logística sobre n-gramas reponderados por Naive Bayes (Wang & Manning, 2012) |
| **Powerset** | Una sola clasificación multiclase sobre la combinación HS-TR-AG (`000`, `111`, `101`...) |
| **Ensamble** | Promedio simple de probabilidades de varios modelos |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


## 1. Modelos base (pipeline del notebook original)
`toxic_base.py` contiene, sin cambios, la normalización, las 38 variables de léxico (H1-H10), los tres bloques (TF-IDF de palabras 1-2, TF-IDF de caracteres `char_wb` 2-5 y léxico) y los cinco modelos base. Los vectorizadores y el escalador se ajustan dentro de cada partición. Las probabilidades OOF quedan en caché en `oof/` para que los notebooks siguientes no reentrenen.

In [2]:
R = {}
OOF = {}
for b in tb.BASES_ORIGINALES:
    OOF[b] = {s: tb.oof(d, b, s) for s in tb.SEEDS}
    R[b] = tb.evaluar(d, OOF[b])
    tb.log('OOF listo:', b)
tb.tabla(R)

[     1s] OOF listo: CNB


[     1s] OOF listo: LR


[     1s] OOF listo: SVM


[     1s] OOF listo: NB-LR


[     1s] OOF listo: Powerset


,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
CNB,0.7509,0.7794,0.7086,0.7463,0.0023,0.7090,0.749 / 0.744 / 0.746
LR,0.7924,0.8246,0.7899,0.8023,0.0020,0.7499,0.803 / 0.805 / 0.800
SVM,0.7890,0.8279,0.7882,0.8017,0.0009,0.7527,0.803 / 0.801 / 0.801
NB-LR,0.7916,0.8231,0.7883,0.8010,0.0009,0.7578,0.801 / 0.802 / 0.800
Powerset,0.7937,0.8284,0.7889,0.8037,0.0021,0.7793,0.806 / 0.804 / 0.801


## 2. Ensamble y stacking

In [3]:
R['Ensamble LR + Powerset'] = tb.evaluar(d, {s: (OOF['LR'][s] + OOF['Powerset'][s]) / 2 for s in tb.SEEDS})
ST = {s: tb.stacking(d, tb.BASES_ORIGINALES, s, con_test=False)[0] for s in tb.SEEDS}
R['Stacking 5 modelos (base enviada)'] = tb.evaluar(d, ST)
tabla_E0 = tb.tabla(R)
tabla_E0

,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
CNB,0.7509,0.7794,0.7086,0.7463,0.0023,0.7090,0.749 / 0.744 / 0.746
LR,0.7924,0.8246,0.7899,0.8023,0.0020,0.7499,0.803 / 0.805 / 0.800
SVM,0.7890,0.8279,0.7882,0.8017,0.0009,0.7527,0.803 / 0.801 / 0.801
NB-LR,0.7916,0.8231,0.7883,0.8010,0.0009,0.7578,0.801 / 0.802 / 0.800
Powerset,0.7937,0.8284,0.7889,0.8037,0.0021,0.7793,0.806 / 0.804 / 0.801
Ensamble LR + Powerset,0.7975,0.8299,0.7921,0.8065,0.0015,0.7680,0.809 / 0.806 / 0.805
Stacking 5 modelos (base enviada),0.8038,0.8345,0.7994,0.8126,0.0021,0.7757,0.815 / 0.810 / 0.813


## 3. Reproducción exacta de la submission 0,79160
Con la semilla 42 (la del notebook original), el stacking debe dar el mismo CSV que `toxic_submission.csv`.

In [4]:
P42, Pt42 = tb.stacking(d, tb.BASES_ORIGINALES, 42)
print('F1 medio OOF semilla 42:', round(tb.metricas(d.Y, tb.binarizar(P42))['F1 medio'], 4))
sub = tb.guardar_submission(d, tb.binarizar(Pt42), 'toxic_submission_E0.csv')

F1 medio OOF semilla 42: 0.8152
[     2s] Guardada /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/experimentos/submissions/toxic_submission_E0.csv


Prevalencia train : {'HS': np.float64(0.437), 'TR': np.float64(0.3), 'AG': np.float64(0.382)}
Prevalencia pred. : {'HS': np.float64(0.393), 'TR': np.float64(0.298), 'AG': np.float64(0.373)}
Cambios frente a la base 0,79160: 0 filas ({'HS': 0, 'TR': 0, 'AG': 0})


## 4. Conclusión
- La reproducción es **exacta**: 0 filas distintas respecto a la submission de 0,79160.
- Con 3 semillas el stacking da **F1 medio ≈ 0,813 ± 0,002**. El 0,815 del notebook original salió de una semilla algo favorable, y esta referencia es más honesta.
- El ranking de modelos se mantiene: Powerset y LR son los mejores individuales, el ensamble suma unos 0,3 puntos y el stacking unos 0,6 más.
- `toxic_submission_E0.csv` es idéntico a la base, así que **no se envía**.